# AEGIS — Análise Exploratória de Dados Orbitais
## NASA EONET (Earth Observatory Natural Event Tracker)

**Global Solution 2026 — Indústria Espacial | 2TSCPW**

Este notebook realiza a análise exploratória dos eventos de desastres naturais rastreados por satélites da NASA.
Os dados são obtidos em tempo real via **NASA EONET v3 API** — open data, sem autenticação.

**Objetivo:** Compreender o volume, distribuição geográfica, padrões temporais e fontes orbitais
dos desastres monitorados por satélite, estabelecendo a base analítica para os modelos preditivos do AEGIS.

**Fontes de dados:**
- NASA EONET v3: `https://eonet.gsfc.nasa.gov/api/v3/`
- Sensores orbitais: MODIS, VIIRS, Copernicus Sentinel, USGS, GDACS

---
## 1. Setup e Importações

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timezone
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Diretório de figuras — robusto tanto em execução interativa quanto via nbconvert
FIG_DIR = Path(os.path.abspath('')).parent / 'notebooks' if 'notebooks' not in os.path.abspath('') else Path(os.path.abspath(''))
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Paleta de cores alinhada ao AEGIS dashboard
PALETTE = {
    'critical':  '#FF6B35',
    'high':      '#FFB547',
    'medium':    '#00D4FF',
    'low':       '#5EE0C2',
    'bg':        '#0a0a0f',
    'surface':   '#12121a',
    'text':      '#e2e8f0',
    'grid':      '#1e293b',
}

CAT_COLORS = {
    'wildfires':      '#FF6B35',
    'severeStorms':   '#00D4FF',
    'floods':         '#5EE0C2',
    'earthquakes':    '#FFB547',
    'volcanoes':      '#FF4444',
    'drought':        '#C4A55A',
    'landslides':     '#8B6914',
    'seaLakeIce':     '#B0E0FF',
    'dustHaze':       '#D4C4A0',
    'snow':           '#FFFFFF',
    'tempExtremes':   '#FF8C42',
    'waterColor':     '#2E86AB',
    'manmade':        '#A23B72',
}

plt.rcParams.update({
    'figure.facecolor': PALETTE['bg'],
    'axes.facecolor':   PALETTE['surface'],
    'axes.edgecolor':   PALETTE['grid'],
    'axes.labelcolor':  PALETTE['text'],
    'xtick.color':      PALETTE['text'],
    'ytick.color':      PALETTE['text'],
    'text.color':       PALETTE['text'],
    'grid.color':       PALETTE['grid'],
    'grid.alpha':       0.5,
    'figure.dpi':       120,
    'font.family':      'monospace',
})

print(f'Ambiente configurado. Figuras serão salvas em: {FIG_DIR}')

---
## 2. Coleta de Dados — NASA EONET v3

A NASA EONET (Earth Observatory Natural Event Tracker) é uma API pública que agrega
eventos de desastres naturais detectados por satélites orbitais como MODIS, VIIRS e Copernicus.

Cada evento registra: tipo de desastre, localização geográfica (lat/lon), magnitude quando disponível,
data de detecção e fonte orbital (qual satélite detectou).

Coletamos três conjuntos:
1. Eventos **abertos** (ativos agora)
2. Eventos **fechados** dos últimos 730 dias (2 anos de histórico)
3. Catálogo de **categorias** disponíveis

In [ ]:
BASE_URL = 'https://eonet.gsfc.nasa.gov/api/v3'

def fetch_events(status='open', limit=200, days=None):
    params = {'status': status, 'limit': limit}
    if days:
        params['days'] = days
    r = requests.get(f'{BASE_URL}/events', params=params, timeout=30)
    r.raise_for_status()
    return r.json()['events']

def fetch_categories():
    r = requests.get(f'{BASE_URL}/categories', timeout=10)
    return r.json()['categories']

print('Coletando eventos abertos...')
open_raw = fetch_events(status='open', limit=200)

print('Coletando histórico (730 dias)...')
closed_raw = fetch_events(status='closed', limit=1000, days=730)

print('Coletando categorias...')
categories_raw = fetch_categories()

print(f'Eventos abertos:  {len(open_raw)}')
print(f'Eventos fechados: {len(closed_raw)}')
print(f'Categorias:       {len(categories_raw)}')

In [ ]:
def normalize_events(raw_list, status_label):
    """Normaliza lista de eventos EONET em DataFrame."""
    rows = []
    for e in raw_list:
        if not e.get('geometry'):
            continue
        geo = e['geometry'][0]  # ponto de detecção inicial
        category = e['categories'][0]['id'] if e['categories'] else 'unknown'
        category_title = e['categories'][0]['title'] if e['categories'] else 'Unknown'
        sources = [s['id'] for s in e.get('sources', [])]

        date_opened = pd.to_datetime(geo['date'], utc=True)
        date_closed = pd.to_datetime(e['closed'], utc=True) if e.get('closed') else None
        duration = (date_closed - date_opened).days if date_closed else None

        rows.append({
            'id':              e['id'],
            'title':           e['title'],
            'category':        category,
            'category_title':  category_title,
            'status':          status_label,
            'date_opened':     date_opened,
            'date_closed':     date_closed,
            'duration_days':   duration,
            'lat':             geo['coordinates'][1],
            'lon':             geo['coordinates'][0],
            'magnitude':       geo.get('magnitudeValue'),
            'magnitude_unit':  geo.get('magnitudeUnit'),
            'n_geometry_pts':  len(e['geometry']),
            'sources':         ', '.join(sources) if sources else 'unknown',
            'n_sources':       len(sources),
        })
    return pd.DataFrame(rows)

df_open   = normalize_events(open_raw, 'open')
df_closed = normalize_events(closed_raw, 'closed')
df = pd.concat([df_open, df_closed], ignore_index=True)
df_cats = pd.DataFrame(categories_raw)

print(f'DataFrame total: {len(df)} eventos | {df.shape[1]} colunas')
print(f'Período: {df["date_opened"].min().date()} → {df["date_opened"].max().date()}')
df.head(3)

---
## 3. Visão Geral dos Dados

In [ ]:
print('=== VISÃO GERAL DO DATASET ===')
print(f'Total de eventos:            {len(df):,}')
print(f'Eventos ativos (abertos):    {len(df_open):,}')
print(f'Eventos históricos:          {len(df_closed):,}')
print(f'Período coberto:             {df["date_opened"].min().date()} a {df["date_opened"].max().date()}')
print(f'Categorias únicas:           {df["category"].nunique()}')
print(f'Fontes orbitais únicas:      {len(set(",".join(df["sources"]).split(", ")))}')
print(f'Eventos com magnitude:       {df["magnitude"].notna().sum():,} ({df["magnitude"].notna().mean()*100:.1f}%)')
print()

# Distribuição por categoria
cat_summary = (
    df.groupby(['category', 'category_title'])
    .agg(total=('id','count'), abertos=('status', lambda x: (x=='open').sum()))
    .reset_index()
    .sort_values('total', ascending=False)
)
cat_summary['pct'] = (cat_summary['total'] / len(df) * 100).round(1)
print('=== EVENTOS POR CATEGORIA ===')
print(cat_summary[['category_title','total','abertos','pct']].to_string(index=False))

---
## 4. Distribuição por Categoria de Desastre

**Metodologia:** Analisamos a frequência de cada tipo de evento para entender quais desastres
são mais monitorados por satélite e quais representam maior volume de alertas. Essa distribuição
informa diretamente quais categorias o AEGIS deve priorizar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Distribuição de Eventos por Categoria — NASA EONET', fontsize=13, y=1.02)

ax = axes[0]
cs = cat_summary.sort_values('total')
colors = [CAT_COLORS.get(c, '#888') for c in cs['category']]
bars = ax.barh(cs['category_title'], cs['total'], color=colors, edgecolor='none')
ax.set_xlabel('Total de eventos (últimos 2 anos)')
ax.set_title('Volume por tipo de desastre')
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, cs['total']):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=8)

ax2 = axes[1]
top6 = cat_summary.nlargest(6, 'total')
x = np.arange(len(top6))
width = 0.38
ax2.bar(x - width/2, top6['total'] - top6['abertos'], width, label='Fechados',
        color='#334155', edgecolor='none')
ax2.bar(x + width/2, top6['abertos'], width, label='Ativos agora',
        color=PALETTE['critical'], edgecolor='none')
ax2.set_xticks(x)
ax2.set_xticklabels([t[:10] for t in top6['category_title']], rotation=30, ha='right', fontsize=8)
ax2.set_title('Top 6 categorias: ativos vs histórico')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_categorias.png', dpi=120, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()
print(f"Insight: {cat_summary.nlargest(2,'total')['category_title'].tolist()} representam "
      f"{cat_summary.nlargest(2,'total')['pct'].sum():.0f}% dos eventos.")

---
## 5. Análise Geográfica

**Metodologia:** Mapeamos todos os eventos por coordenadas geográficas (lat/lon fornecidas pela API EONET,
originadas de telemetria orbital) para identificar hotspots globais de risco.
Usamos Plotly para uma visualização interativa do mapa mundial.

In [ ]:
# Mapa global interativo
df_map = df.dropna(subset=['lat','lon']).copy()
df_map['color'] = df_map['category'].map(CAT_COLORS).fillna('#888888')
df_map['size'] = df_map['status'].map({'open': 8, 'closed': 4})
df_map['label'] = df_map['title'].str[:40]

fig_map = px.scatter_geo(
    df_map,
    lat='lat', lon='lon',
    color='category_title',
    size='size',
    hover_name='label',
    hover_data={'lat': ':.2f', 'lon': ':.2f', 'status': True, 'date_opened': True},
    color_discrete_map={row['category_title']: CAT_COLORS.get(row['category'], '#888')
                        for _, row in cat_summary.iterrows()},
    projection='natural earth',
    title='Eventos de Desastres Naturais Rastreados por Satélite (NASA EONET)',
)
fig_map.update_layout(
    paper_bgcolor='#0a0a0f',
    geo=dict(
        bgcolor='#12121a',
        landcolor='#1e2a3a',
        oceancolor='#0a1628',
        showocean=True,
        showland=True,
        showcountries=True,
        countrycolor='#334155',
        framecolor='#334155',
    ),
    font_color='#e2e8f0',
    legend_title_text='Categoria',
    height=500,
)
fig_map.show()

In [ ]:
df_map['lat_bin'] = (df_map['lat'] // 15) * 15
df_map['lon_bin'] = (df_map['lon'] // 15) * 15

density = df_map.groupby(['lat_bin','lon_bin']).size().reset_index(name='count')
density_pivot = density.pivot(index='lat_bin', columns='lon_bin', values='count').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('Densidade Geográfica de Eventos — Grade 15°×15°', fontsize=12)
sns.heatmap(density_pivot, cmap='YlOrRd', linewidths=0.3, linecolor='#1e293b',
            ax=ax, cbar_kws={'label': 'Nº de eventos'})
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_title('Hotspots globais detectados por satélite')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_densidade_geo.png', dpi=120, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()

print('Top 5 hotspots geográficos:')
print(density.nlargest(5,'count')[['lat_bin','lon_bin','count']].to_string(index=False))

---
## 6. Análise Temporal

**Metodologia:** Analisamos a evolução temporal dos eventos ao longo dos 2 anos de histórico.
Séries temporais mensais revelam tendências de aumento/redução e padrões de sazonalidade
por tipo de desastre — informação essencial para modelos preditivos como LSTM e Prophet.

In [ ]:
df['year_month'] = df['date_opened'].dt.to_period('M')
monthly = df.groupby('year_month').size().reset_index(name='count')
monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()
monthly['ma3'] = monthly['count'].rolling(3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly['year_month_dt'], monthly['count'],
                alpha=0.3, color=PALETTE['medium'], label='Eventos/mês')
ax.plot(monthly['year_month_dt'], monthly['count'],
        color=PALETTE['medium'], linewidth=1, alpha=0.8)
ax.plot(monthly['year_month_dt'], monthly['ma3'],
        color=PALETTE['critical'], linewidth=2.5, label='Média móvel 3 meses', linestyle='--')
ax.set_title('Eventos de Desastre Detectados por Satélite — Série Temporal Mensal')
ax.set_xlabel('Mês')
ax.set_ylabel('Número de eventos')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_serie_temporal.png', dpi=120, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()

x = np.arange(len(monthly))
coef = np.polyfit(x, monthly['count'].fillna(0), 1)
tendencia = 'crescente' if coef[0] > 0 else 'decrescente'
print(f'Tendência geral: {tendencia} ({coef[0]:+.2f} eventos/mês)')

In [ ]:
df['month'] = df['date_opened'].dt.month
top6_cats = cat_summary.nlargest(6, 'total')['category'].tolist()
df_top6 = df[df['category'].isin(top6_cats)]

season_pivot = (
    df_top6.groupby(['month', 'category_title'])
    .size()
    .unstack(fill_value=0)
)
month_labels = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
season_pivot.index = month_labels[:len(season_pivot)]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(season_pivot.T, cmap='YlOrRd', linewidths=0.5, linecolor='#1e293b',
            ax=ax, annot=True, fmt='d', cbar_kws={'label': 'Eventos'}, annot_kws={'size': 8})
ax.set_title('Sazonalidade por Tipo de Desastre (Mês × Categoria)')
ax.set_xlabel('Mês')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_sazonalidade.png', dpi=120, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()

---
## 7. Magnitude e Intensidade dos Eventos

**Metodologia:** Quando disponível, a API EONET registra a magnitude do evento (área queimada em acres
para incêndios, magnitude Richter para terremotos). Essa análise quantifica a intensidade
dos eventos além da simples contagem — informação relevante para o score de risco do AEGIS.

In [ ]:
df_mag = df.dropna(subset=['magnitude']).copy()
print(f'Eventos com magnitude registrada: {len(df_mag)} ({len(df_mag)/len(df)*100:.1f}%)')
print()
print('Magnitude por unidade:')
print(df_mag.groupby('magnitude_unit')['magnitude'].describe().round(2))

In [ ]:
units = df_mag['magnitude_unit'].value_counts()
n_units = min(len(units), 3)

fig, axes = plt.subplots(1, n_units, figsize=(5 * n_units, 5))
if n_units == 1:
    axes = [axes]
fig.suptitle('Distribuição de Magnitude por Unidade de Medida', fontsize=12)

for ax, unit in zip(axes, units.index[:n_units]):
    subset = df_mag[df_mag['magnitude_unit'] == unit]['magnitude']
    color = CAT_COLORS.get(
        df_mag[df_mag['magnitude_unit'] == unit]['category'].mode()[0],
        PALETTE['medium']
    )
    ax.hist(subset, bins=30, color=color, edgecolor='none', alpha=0.85)
    ax.set_title(f'{unit}\n(n={len(subset)})')
    ax.set_xlabel(f'Magnitude ({unit})')
    ax.set_ylabel('Frequência')
    ax.axvline(subset.median(), color='white', linestyle='--', alpha=0.6,
               label=f'Mediana: {subset.median():.1f}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_magnitude.png', dpi=120, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()

---
## 8. Fontes Orbitais — Quais Satélites Detectam os Eventos

**Metodologia:** A API EONET registra a fonte de detecção de cada evento — o sensor ou sistema orbital
que identificou o desastre. Essa análise demonstra a infraestrutura espacial que sustenta o AEGIS
e conecta diretamente ao tema da Indústria Espacial da GS 2026.

In [ ]:
# Expandir fontes (cada evento pode ter múltiplas fontes)
df_sources = df[df['sources'] != 'unknown'].copy()
sources_expanded = (
    df_sources['sources']
    .str.split(', ')
    .explode()
    .value_counts()
    .reset_index()
)
sources_expanded.columns = ['source', 'count']

# Descrição dos principais sensores orbitais
SOURCE_INFO = {
    'MODIS_NRT': 'MODIS/NASA — detecção térmica de incêndios e anomalias',
    'VIIRS_NOAA20_NRT': 'VIIRS/NOAA-20 — imageamento noturno e incêndios',
    'VIIRS_SNPP_NRT': 'VIIRS/Suomi-NPP — cobertura global diária',
    'USGS_EHP': 'USGS — sismologia e terremotos',
    'GDACS': 'GDACS/ONU — alerta global de desastres',
    'PDC': 'PDC — Pacific Disaster Center',
    'IRWIN': 'IRWIN/DOI — incêndios florestais EUA',
    'ReliefWeb': 'ReliefWeb/UNOCHA — resposta humanitária',
    'CALFIRE': 'CAL FIRE — incêndios na Califórnia',
    'AVO': 'Alaska Volcano Observatory',
}

sources_expanded['description'] = sources_expanded['source'].map(
    lambda s: SOURCE_INFO.get(s, 'Sistema de monitoramento especializado')
)

print('Fontes orbitais por volume de eventos:')
print(sources_expanded.head(10).to_string(index=False))

In [ ]:
top_sources = sources_expanded.head(8)

fig, ax = plt.subplots(figsize=(12, 5))
colors_src = [PALETTE['critical'], PALETTE['high'], PALETTE['medium'], PALETTE['low'],
              '#8B5CF6', '#EC4899', '#F59E0B', '#10B981']
bars = ax.barh(top_sources['source'][::-1], top_sources['count'][::-1],
               color=colors_src[:len(top_sources)], edgecolor='none')
ax.set_xlabel('Número de eventos detectados')
ax.set_title('Top Fontes Orbitais — Satélites e Sistemas que Alimentam o AEGIS')
ax.grid(axis='x', alpha=0.3)
for bar, (_, row) in zip(bars, top_sources.iloc[::-1].iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{row['count']}", va='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_fontes_orbitais.png', dpi=120, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()

---
## 9. Insights para o AEGIS

**Metodologia:** Identificamos eventos de longa duração (sem resolução) como indicadores de
risco sustentado — exatamente o cenário que o AEGIS precisa monitorar continuamente.
Também construímos a base de features para os modelos preditivos futuros.

In [ ]:
# Duração média por categoria
duration_stats = (
    df.dropna(subset=['duration_days'])
    .groupby('category_title')['duration_days']
    .agg(['mean','median','max','count'])
    .round(1)
    .sort_values('mean', ascending=False)
    .reset_index()
)
print('Duração média dos eventos por categoria (dias):')
print(duration_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
dur_sorted = duration_stats.sort_values('mean')
colors_dur = [CAT_COLORS.get(
    cat_summary[cat_summary['category_title'] == t]['category'].values[0]
    if len(cat_summary[cat_summary['category_title'] == t]) > 0 else 'unknown', '#888')
    for t in dur_sorted['category_title']]
ax.barh(dur_sorted['category_title'], dur_sorted['mean'], color=colors_dur, edgecolor='none')
ax.set_xlabel('Duração média (dias)')
ax.set_title('Persistência Média por Tipo de Evento')
ax.grid(axis='x', alpha=0.3)

ax2 = axes[1]
df_fe = df.copy()
df_fe['lat_zone'] = pd.cut(df_fe['lat'], bins=[-90,-60,-30,0,30,60,90],
                            labels=['Ant','Sul extra','Sul trop','Norte trop','Norte extra','Ártico'])
zone_cat = df_fe.groupby(['lat_zone','category']).size().unstack(fill_value=0)
top4_cats = cat_summary.nlargest(4,'total')['category'].tolist()
zone_cat = zone_cat[[c for c in top4_cats if c in zone_cat.columns]]
zone_cat.plot(kind='bar', ax=ax2, color=[CAT_COLORS.get(c,'#888') for c in zone_cat.columns],
              edgecolor='none', legend=True)
ax2.set_title('Distribuição por Zona Latitudinal (Feature ML)')
ax2.set_xlabel('Zona')
ax2.set_ylabel('Eventos')
ax2.legend(fontsize=7)
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_insights.png', dpi=120, bbox_inches='tight', facecolor=PALETTE['bg'])
plt.show()

In [ ]:
# Feature engineering — features candidatas para o modelo ML
df_features = df.copy()
df_features['month']       = df_features['date_opened'].dt.month
df_features['day_of_week'] = df_features['date_opened'].dt.dayofweek
df_features['year']        = df_features['date_opened'].dt.year
df_features['lat_bin']     = (df_features['lat'] // 10) * 10
df_features['lon_bin']     = (df_features['lon'] // 10) * 10
df_features['has_mag']     = df_features['magnitude'].notna().astype(int)
df_features['multi_source']= (df_features['n_sources'] > 1).astype(int)

print('Features candidatas para os modelos ML do AEGIS:')
feature_cols = ['month','day_of_week','year','lat_bin','lon_bin',
                'has_mag','multi_source','n_sources','n_geometry_pts']
print(df_features[feature_cols].describe().round(2).to_string())

---
## 10. Conclusões e Próximos Passos

**Principais achados desta análise exploratória:**

In [ ]:
top2_cats = cat_summary.nlargest(2,'total')['category_title'].tolist()
top_pct   = cat_summary.nlargest(2,'total')['pct'].sum()
trend_dir = 'crescente' if coef[0] > 0 else 'decrescente'

print('=== INSIGHTS CHAVE ===')
print()
print(f'1. VOLUME: {len(df):,} eventos rastreados por satélite nos últimos 2 anos.')
print(f'   {len(df_open)} eventos ativos agora — monitoramento em tempo real demonstrado.')
print()
print(f'2. CONCENTRAÇÃO: {top2_cats[0]} e {top2_cats[1]} representam '
      f'{top_pct:.0f}% do total de eventos.')
print(f'   O AEGIS deve priorizar essas duas categorias na interface e nos modelos.')
print()
print(f'3. TENDÊNCIA: volume de eventos {trend_dir} ({coef[0]:+.2f} eventos/mês).')
print(f'   Padrão compatível com aumento de eventos climáticos extremos documentado pelo IPCC.')
print()
print(f'4. SAZONALIDADE: incêndios pico em mai–out (verão hemisfério norte),')
print(f'   tempestades distribuídas com pico em ago–set (temporada de furacões).')
print(f'   Padrão previsível → LSTM e Prophet são adequados para forecasting.')
print()
print(f'5. INFRAESTRUTURA ORBITAL: eventos detectados por {len(sources_expanded)} '
      f'fontes distintas.')
print(f'   MODIS, VIIRS e USGS respondem pela maioria — sensores com cobertura global diária.')
print()
print('=== PRÓXIMOS PASSOS (Sprints GS) ===')
print()
print('Sprint ML:')
print('  - Treinar LSTM com série mensal de eventos (target: volume D+1 a D+7)')
print('  - Treinar XGBoost com features: month, lat_bin, lon_bin, category → risk score')
print('  - SHAP para explicabilidade das features mais importantes')
print()
print('Sprint Backend:')
print('  - FastAPI expondo /api/events (EONET + ML score)')
print('  - Endpoint /api/forecast/{category} (LSTM D+1 a D+7)')
print()
print('Sprint Dados:')
print('  - Adicionar Copernicus Sentinel (imagens ópticas para incêndios/enchentes)')
print('  - BDQueimadas INPE (foco específico Brasil)')
print('  - USGS Earthquake Catalog (histórico sísmico 1900–2026)')